### Analytical derivation of decay behavior

Kernel definitions used in this notebook:
$$
\begin{aligned}
\tilde{K}_{\mathrm{IPR}} &= \frac{1}{2^L}\frac{I^{(Z)} + I^{(X)}}{\sqrt{4^{-L} + \left(I^{(Z)}\right)^2}\sqrt{4^{-L} + \left(I^{(X)}\right)^2}} \\
\tilde{K}_{\mathrm{HS}}  &= \frac{1}{2^L}\frac{Z^{(Z)}_\beta Z^{(X)}_\beta}{\sqrt{Z^{(Z)}_{2\beta}}\sqrt{Z^{(X)}_{2\beta}}}.
\end{aligned}
$$

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.size": 11,
    "axes.labelsize": 15,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 12,
})


def Z_x(beta, L):
    beta = np.asarray(beta, dtype=float)
    return (1.0 + np.exp(-2.0 * beta)) ** L


def Z_z(beta, L):
    beta = np.asarray(beta, dtype=float)
    e = np.exp(-2.0 * beta)
    return (1.0 + e) ** L + (1.0 - e) ** L


def I_z(beta, L):
    return Z_z(2.0 * beta, L) / (Z_z(beta, L) ** 2)


def I_x(beta, L):
    return Z_x(2.0 * beta, L) / (Z_x(beta, L) ** 2)


def K_ipr(beta, L):
    iz = I_z(beta, L)
    ix = I_x(beta, L)
    k_raw = (2.0 ** (-L)) * (iz + ix)
    denom = np.sqrt(4.0 ** (-L) + iz ** 2) * np.sqrt(4.0 ** (-L) + ix ** 2)
    return k_raw / denom


def K_hs(beta, L):
    beta = np.asarray(beta, dtype=float)
    numerator = Z_z(beta, L) * Z_x(beta, L)
    denominator = np.sqrt(Z_z(2.0 * beta, L)) * np.sqrt(Z_x(2.0 * beta, L))
    return (2.0 ** (-L)) * numerator / denominator


IPR_COLOR = "#1b9e77"
HS_COLOR = "#d95f02"


def plot_analytic_kernels(
    L_values=(10, 50, 300),
    beta_min=0.0,
    beta_max=3.0,
    n_beta=500,
    save_path=None,
):
    beta = np.linspace(beta_min, beta_max, n_beta)

    fig, axes = plt.subplots(1, len(L_values), figsize=(12, 4), sharey=True)
    axes = np.atleast_1d(axes)

    for ax, L in zip(axes, L_values):
        k_ipr = K_ipr(beta, L)
        k_hs = K_hs(beta, L)

        ax.plot(beta, k_ipr, lw=2, color=IPR_COLOR, label=r"$\tilde{K}_{\mathrm{IPR}}$")
        ax.plot(beta, k_hs, lw=2, color=HS_COLOR, label=r"$\tilde{K}_{\mathrm{HS}}$")
        ax.text(0.95, 0.92, rf"$L={L}$", transform=ax.transAxes, ha="right", va="top", fontsize=15)
        ax.set_xlabel(r"$\beta$")
        ax.set_ylim(-0.02, 1.05)
        ax.grid(alpha=0.3)

    axes[0].set_ylabel("Kernel value")
    axes[-1].legend()
    fig.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        fig.savefig(save_path, bbox_inches="tight")

    plt.show()

In [ ]:
plot_analytic_kernels(
    L_values=(10, 50, 300),
    beta_min=0.0,
    beta_max=1.0,
    n_beta=500,
    save_path=Path("Paper plot analytical kernel") / "analytical_kernel_comparison.pdf",
)